In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           INSTRUCTIONS — LIRE AVANT D'EXÉCUTER              ║
# ╚══════════════════════════════════════════════════════════════╝
#
# ÉTAPE 1 — KAGGLE DATASETS
# ─────────────────────────
# Clique sur "Add Data" en haut à droite et ajoute ces 2 datasets :
#   • dataaaa       → contient le modèle LoRA (psychollm_final)
#   • ragrag        → contient l'index FAISS (index.faiss + index.pkl)
# Les deux datasets appartiennent au compte Kaggle : nourmniff
#
# ÉTAPE 2 — GPU
# ─────────────
# Settings (en haut à droite) → Accelerator → GPU T4 x2
# Sans GPU le modèle ne peut pas tourner.
#
# ÉTAPE 3 — NGROK TOKEN
# ──────────────────────
# 1. Crée un compte GRATUIT sur : https://ngrok.com
# 2. Va sur : https://dashboard.ngrok.com/get-started/your-authtoken
# 3. Copie ton token
# 4. Dans la cellule "Exposer via ngrok", remplace :
#       NGROK_TOKEN = "TON_TOKEN_NGROK_ICI"
#    par ton vrai token.
#
# ÉTAPE 4 — IP DE TON PC
# ───────────────────────
# Dans la DERNIÈRE cellule du notebook, remplace :
#       FASTAPI_URL = "http://TON_IP:8000"
# par l'IP locale du PC qui fait tourner main.py
# Pour trouver ton IP : ouvre cmd → tape ipconfig → cherche IPv4
#
# ÉTAPE 5 — EXÉCUTER
# ───────────────────
# Clique sur "Run All" et attends ~10 minutes le temps
# que le modèle se charge.
# À la fin tu verras l'URL ngrok s'afficher.
#
# ⚠️  IMPORTANT
# ─────────────
# • Ne partage JAMAIS ton token ngrok avec personne
# • Chaque session Kaggle dure max 12h puis s'arrête
# • Si le notebook s'arrête, relance-le — l'URL changera
#   mais main.py se met à jour automatiquement

In [ ]:
%%capture

!pip install -q peft transformers accelerate bitsandbytes
!pip install -q faiss-cpu langchain langchain-community sentence-transformers
!pip install -q flask flask-cors pyngrok

print("✅ Installation terminée")

In [ ]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

from peft import PeftModel

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


# ─────────────────────────────
# Chemins
# ─────────────────────────────

ADAPTER_PATH = "/kaggle/input/datasets/nourmniff/dataaaa/kaggle/working/psychollm_final"

FAISS_PATH = "/kaggle/input/datasets/nourmniff/ragrag"

BASE_MODEL = "GMLHUHE/PsyLLM-8B"


print("🔄 Chargement du modèle...")


# Quantization 4bit

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)


base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)


tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH
)

tokenizer.pad_token = tokenizer.eos_token


model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()

print("✅ Modèle chargé")


# ─────────────────────────────
# Charger le RAG
# ─────────────────────────────

print("🔄 Chargement FAISS...")


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={
        "device": "cpu"
    }
)


vectorstore = FAISS.load_local(
    folder_path=FAISS_PATH,
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

print("✅ RAG chargé")

In [ ]:
SYSTEM_PROMPT = """
You are PsychoLLM, an expert clinical psychiatry assistant
specializing in Schizophrenia, Bipolar Disorder,
and Personality Disorders (BPD/STPD).

You engage in rigorous clinical debates
and differential diagnosis reasoning
with DSM-5 precision.
"""


def retrieve_context(query: str, k: int = 4):

    docs = vectorstore.similarity_search(
        query,
        k=k
    )

    parts = []

    for i, doc in enumerate(docs, 1):

        source = doc.metadata.get(
            "source",
            f"Ref {i}"
        )

        parts.append(
            f"[{source}]\n{doc.page_content}"
        )

    return "\n\n".join(parts)


def build_prompt(user_input, context, history):

    prompt = (
        f"<|im_start|>system\n"
        f"{SYSTEM_PROMPT}"
        f"<|im_end|>\n"
    )

    for turn in history:

        prompt += (
            f"<|im_start|>{turn['role']}\n"
            f"{turn['content']}"
            f"<|im_end|>\n"
        )

    if not history:

        user_msg = (
            f"[CLINICAL LITERATURE]\n"
            f"{context}\n\n"
            f"[CLINICIAN]\n"
            f"{user_input}"
        )

    else:

        user_msg = user_input

    prompt += (
        f"<|im_start|>user\n"
        f"{user_msg}"
        f"<|im_end|>\n"
    )

    prompt += "<|im_start|>assistant\n"

    return prompt


def generate_response(
    prompt,
    max_new_tokens=512
):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3000
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


print("✅ Pipeline prêt")

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS

import uuid


app = Flask(__name__)

CORS(app)


sessions = {}


@app.route("/health", methods=["GET"])
def health():

    return jsonify({
        "status": "ok",
        "model": "PsychoLLM"
    })


@app.route("/chat", methods=["POST"])
def chat():

    try:

        data = request.get_json()

        message = data.get(
            "message",
            ""
        )

        session_id = data.get(
            "session_id",
            str(uuid.uuid4())
        )

        if not message:

            return jsonify({
                "error": "message requis"
            }), 400

        if session_id not in sessions:

            sessions[session_id] = []

        history = sessions[session_id]

        context = retrieve_context(
            message
        )

        prompt = build_prompt(
            message,
            context,
            history
        )

        response = generate_response(
            prompt
        )

        history.append({
            "role": "user",
            "content": message
        })

        history.append({
            "role": "assistant",
            "content": response
        })

        if len(history) > 20:

            history = history[-20:]

        sessions[session_id] = history

        return jsonify({

            "response": response,

            "session_id": session_id,

            "turn": len(history) // 2
        })

    except Exception as e:

        return jsonify({

            "error": str(e)

        }), 500


@app.route(
    "/session/<session_id>",
    methods=["DELETE"]
)
def reset_session(session_id):

    if session_id in sessions:

        del sessions[session_id]

    return jsonify({

        "status": "reset"

    })


@app.route(
    "/session/<session_id>",
    methods=["GET"]
)
def get_history(session_id):

    history = sessions.get(
        session_id,
        []
    )

    return jsonify({

        "history": history,

        "turns": len(history) // 2
    })


print("✅ Serveur Flask prêt")

In [ ]:
from pyngrok import ngrok
import threading

# IMPORTANT — mets ton vrai token ngrok
NGROK_TOKEN = "TON_TOKEN_NGROK_ICI"

PORT = 5001

ngrok.set_auth_token(NGROK_TOKEN)

def run_flask():

    app.run(
        host="0.0.0.0",
        port=PORT,
        debug=False,
        use_reloader=False
    )

flask_thread = threading.Thread(
    target=run_flask,
    daemon=True
)

flask_thread.start()

public_url = ngrok.connect(PORT)

print("\n" + "="*50)
print("✅ API PsychoLLM disponible sur :")
print(public_url)
print("="*50)

print("\nEndpoints :")

print(f"{public_url}/health")
print(f"{public_url}/chat")

In [ ]:
# ── Envoyer la nouvelle URL au FastAPI automatiquement ──

import requests

FASTAPI_URL = "http://TON_IP:8000"  # IP de ton PC
response = requests.post(

    f"{FASTAPI_URL}/update-url",

    json={
        "url": public_url.public_url
    }

)

print("✅ FastAPI mis à jour:")
print(response.json())